# Watch two trained models play

Loads a white and a black checkpoint, plays one game, then lets you scrub through it.
For each position you see the board, the move played (SAN), the mover's value output,
the material balance from the mover's side, and **value + material**: under
potential-based shaping the value head learns `V - material`, so add material back
before reading whether the model thinks it is winning (see `learning/records/0003`).

In [ ]:
import torch
from src.model.chess_model import ChessPolicyProbs
from src.viz.play import play_game

CKPT_DIR = "experiments/potential-based-shaping"
WHITE_CKPT = f"{CKPT_DIR}/white_model_20260915153257_episodes_500.pth"
BLACK_CKPT = f"{CKPT_DIR}/black_model_20260915153257_episodes_500.pth"

def load_model(path):
    model = ChessPolicyProbs()
    model.load_state_dict(torch.load(path, map_location="cpu"))
    return model.eval()

white_model = load_model(WHITE_CKPT)
black_model = load_model(BLACK_CKPT)

In [ ]:
# greedy=True plays the most likely move every time; False samples from the policy.
game = play_game(white_model, black_model, greedy=False, max_moves=300, seed=0)
print(f"{len(game.moves)} moves, result: {game.result}")

In [ ]:
import chess, chess.svg
import ipywidgets as widgets
from IPython.display import display, HTML

def show(i):
    if i == 0:
        board = chess.Board()
        info = "<b>Start position</b>"
    else:
        rec = game.moves[i - 1]
        board = chess.Board(rec.fen_after)
        top = ", ".join(f"{s} {p:.2f}" for s, p in rec.top_moves)
        info = (
            f"<b>Move {rec.move_number}</b> {rec.side} plays <b>{rec.san}</b> "
            f"(p={rec.prob_played:.2f})<br>"
            f"value = {rec.value:+.2f} &nbsp; material = {rec.material:+.0f} &nbsp; "
            f"<b>value + material = {rec.value_plus_material:+.2f}</b><br>"
            f"top-3: {top}"
        )
    last = board.peek() if board.move_stack else None
    svg = chess.svg.board(board, size=400, lastmove=last)
    display(HTML(f"<div style='display:flex;gap:24px;align-items:flex-start'>"
                 f"<div>{svg}</div><div style='font-family:monospace'>{info}<br><br>"
                 f"result: {game.result}</div></div>"))

slider = widgets.IntSlider(min=0, max=len(game.moves), step=1, value=0, description="ply",
                           continuous_update=False, layout=widgets.Layout(width="600px"))
widgets.interact(show, i=slider);

In [ ]:
# Value vs value+material over the game, per side.
import matplotlib.pyplot as plt

for side, color in (("white", "tab:blue"), ("black", "tab:red")):
    recs = [m for m in game.moves if m.side == side]
    x = [m.move_number for m in recs]
    plt.plot(x, [m.value for m in recs], color=color, alpha=0.4, label=f"{side} value (raw)")
    plt.plot(x, [m.value_plus_material for m in recs], color=color, label=f"{side} value + material")
plt.axhline(0, color="gray", lw=0.5)
plt.xlabel("ply"); plt.ylabel("mover's perspective"); plt.legend(); plt.title(f"result: {game.result}")
plt.show()